In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import math

# Load pre-trained model and tokenizer
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Encode the sentence
sentence = "I love you"
inputs = tokenizer(sentence, return_tensors="pt")

# Compute log probabilities
with torch.no_grad():
    outputs = model(**inputs, labels=inputs["input_ids"])
    log_likelihood = -outputs.loss.item() * inputs["input_ids"].size(1)
    sentence_probability = math.exp(log_likelihood)

print(f"Log Probability: {log_likelihood}")
print(f"Sentence Probability: {sentence_probability}")

`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Log Probability: -14.55488920211792
Sentence Probability: 4.774104651637064e-07


In [2]:
logits = outputs.logits

In [3]:
input_ids = inputs["input_ids"]

In [4]:
input_ids

tensor([[  40, 1842,  345]])

In [5]:
shift_logits = logits[:, :-1, :].contiguous()

In [6]:
shift_logits

tensor([[[-39.3084, -39.0100, -41.8374,  ..., -46.9337, -44.9073, -39.5149],
         [-84.4961, -85.0686, -90.6138,  ..., -91.6659, -93.1035, -87.5303]]])

In [7]:
shift_labels = input_ids[:, 1:].contiguous()

In [8]:
shift_labels

tensor([[1842,  345]])

In [9]:
import torch.nn.functional as F

log_probs = F.log_softmax(shift_logits, dim=-1)

In [10]:
log_probs

tensor([[[ -6.4495,  -6.1511,  -8.9785,  ..., -14.0748, -12.0484,  -6.6560],
         [ -8.3050,  -8.8775, -14.4227,  ..., -15.4748, -16.9124, -11.3392]]])

In [11]:
log_probs_for_tokens = log_probs.gather(2, shift_labels.unsqueeze(-1)).squeeze(-1)

In [12]:
log_probs_for_tokens

tensor([[-7.3372, -2.3660]])

In [13]:
total_log_prob = log_probs_for_tokens.sum().item()

In [14]:
total_log_prob

-9.703259468078613

In [15]:
sentence_probability = torch.exp(torch.tensor(total_log_prob)).item()

In [16]:
sentence_probability

6.108406523708254e-05

In [17]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import math
import torch.nn.functional as F

model_name = "meta-llama/Meta-Llama-3-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto", torch_dtype=torch.float16)

def cal_part_logprob(input_text: str, _tokenizer):

    # Load pre-trained model and tokenizer
    # model_name = "gpt2"

    # model = AutoModelForCausalLM.from_pretrained(model_name)
    # model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto", torch_dtype=torch.float16)

    
    # Encode the sentence
    sentence = input_text
    inputs = _tokenizer(sentence, return_tensors="pt")
    
    # Compute log probabilities
    with torch.no_grad():
        outputs = model(**inputs, labels=inputs["input_ids"], loss_type='ForCausalLMLoss')

    logits = outputs.logits
    input_ids = inputs["input_ids"]
    shift_logits = logits[:, :-1, :].contiguous()
    shift_labels = input_ids[:, 1:].contiguous()

    log_probs = F.log_softmax(shift_logits, dim=-1)
    log_probs_for_tokens = log_probs.gather(2, shift_labels.unsqueeze(-1)).squeeze(-1)
    total_log_prob = log_probs_for_tokens.sum().item()
    avg_logProb = total_log_prob/len(input_ids[0])

    return log_probs_for_tokens, total_log_prob, avg_logProb
    
    # sentence_probability = torch.exp(torch.tensor(total_log_prob)).item()

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [18]:
cal_part_logprob('I love you', tokenizer)

(tensor([[-2.9941, -3.1719, -4.5234]], dtype=torch.float16),
 -10.6875,
 -2.671875)

##### Idea: calcuate the average logprob per token in a sentence, passage, or a multi-passage passage, it seems not difficult to calculate

In [19]:
import sys
import os
notebook_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(notebook_dir, '..'))
analysis_root = os.path.abspath(os.path.join(notebook_dir, '..', 'analysis'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
if analysis_root not in sys.path:
    sys.path.insert(0, analysis_root)

from analysis.tools import coherence_cal

In [20]:
res, doc_dict, doc_length_dict = coherence_cal.get_res_and_dicts('nq_test', 'mt5')

In [22]:
for qid_no in range(10, 20):
    qid = 'test_'+str(qid_no)
    print(qid)
    doc_texts = res[(res.qid==qid)&(res['rank']<3)].docno.apply(lambda x: doc_dict[str(x)])
    
    for text in doc_texts:
        _, _, avg_logProb = cal_part_logprob(text, tokenizer)
        print(avg_logProb)
    text = ''.join(doc_texts)
    # text = text[:-1]
    _, _, avg_logProb = cal_part_logprob(text, tokenizer)
    print(avg_logProb)

test_10
-2.4965753424657535
-2.736328125
-2.5606617647058822
-2.471813725490196
test_11
-2.1194581280788176
-1.9722222222222223
-2.874031007751938
-2.3161157024793386
test_12
-2.7775735294117645
-1.780674846625767
-2.8535714285714286
-2.3123569794050343
test_13
-2.1241379310344826
-2.587686567164179
-2.653361344537815
-2.3295454545454546
test_14
-2.84
-2.6666666666666665
-2.4834710743801653
-2.5904255319148937
test_15
-3.6449579831932772
-3.482300884955752
-2.752
-3.259154929577465
test_16
-2.109375
-2.1636690647482015
-3.511111111111111
-2.072704081632653
test_17
-2.8461538461538463
-2.17
-2.385658914728682
-2.483739837398374
test_18
-1.8597122302158273
-1.8172348484848484
-2.2083333333333335
-1.4898477157360406
test_19
-2.4510869565217392
-1.8082191780821917
-1.9557692307692307
-2.032766990291262


-3.612576392389113


In [76]:
import numpy as np

np.random.seed(100)
rand_scores = np.random.rand(10)
points = np.array(range(10))
# weights = [
weights = np.array([1.0]*len(points)) # uniform
# weights = np.exp(-points) # top-heavy
# weights = np.exp(points+1-len(points)) # tail-heavy
# weights = np.exp((2*points/(len(points)-1)-1)**2) # lost-in-the-middle

print(type(weights), weights)
np.sum(weights*np.mean(rand_scores))/np.sum(weights)

<class 'numpy.ndarray'> [1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


0.4425757785871915

In [72]:
np.mean(rand_scores)

0.4425757785871915

In [56]:
np.random.rand(10)

array([0.89132195, 0.20920212, 0.18532822, 0.10837689, 0.21969749,
       0.97862378, 0.81168315, 0.17194101, 0.81622475, 0.27407375])

In [57]:
np.exp((2*points/(len(points)-1)-1)**2)

array([2.71828183, 1.83113917, 1.36157481, 1.11751907, 1.0124222 ,
       1.0124222 , 1.11751907, 1.36157481, 1.83113917, 2.71828183])